# Systematic Model Comparison for Classification
## Module 2, Session 6: Comparing 6+ Algorithms to Find the Best Classifier

**Goal:** Systematically compare multiple classification algorithms  
**Dataset:** Heart disease prediction dataset  
**Problem:** Binary classification with medical data  
**Target:** Find the best algorithm and understand when to use which  

---

### What You'll Learn
1. Systematic model comparison methodology
2. Training 6+ classification algorithms
3. ROC curves and AUC scores
4. Creating comprehensive comparison tables
5. Understanding algorithm strengths and weaknesses
6. Building a decision guide: when to use which algorithm

---

**Created:** 2026-01-06  
**Course:** ML for Engineers  
**Module:** 2 - Classification

## Step 1: Import Libraries

In [ ]:
# Data manipulation
import pandas as pd
import numpy as np
import time

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score

# Machine Learning Algorithms
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier
)
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

# Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Settings
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
np.random.seed(42)

print("✓ All libraries imported successfully!")

## Step 2: Create Synthetic Heart Disease Dataset

We'll create a realistic heart disease prediction dataset.

> **AI Assistant Prompt:**  
> "Create a synthetic heart disease dataset with 800 patients. Include features: age, resting blood pressure, cholesterol, max heart rate, chest pain type, and fasting blood sugar. Make disease probability increase with age, high cholesterol, and high blood pressure."

In [ ]:
# Create synthetic heart disease dataset
np.random.seed(42)
n_patients = 800

data = {
    'age': np.random.randint(30, 80, n_patients),
    'sex': np.random.choice([0, 1], n_patients, p=[0.4, 0.6]),  # 0=female, 1=male
    'chest_pain_type': np.random.choice([0, 1, 2, 3], n_patients),
    'resting_bp': np.random.randint(90, 200, n_patients),
    'cholesterol': np.random.randint(120, 350, n_patients),
    'fasting_blood_sugar': np.random.choice([0, 1], n_patients, p=[0.7, 0.3]),
    'max_heart_rate': np.random.randint(80, 200, n_patients),
    'exercise_angina': np.random.choice([0, 1], n_patients, p=[0.6, 0.4]),
    'oldpeak': np.random.uniform(0, 6, n_patients),  # ST depression
}

df = pd.DataFrame(data)

# Create target based on risk factors
risk_score = (
    0.3 * ((df['age'] - 30) / 50) +                    # Age risk
    0.2 * ((df['cholesterol'] - 120) / 230) +          # Cholesterol
    0.2 * ((df['resting_bp'] - 90) / 110) +            # Blood pressure
    0.1 * (1 - (df['max_heart_rate'] - 80) / 120) +    # Low max HR is bad
    0.1 * df['exercise_angina'] +                      # Exercise angina
    0.1 * df['oldpeak'] / 6 +                          # ST depression
    np.random.normal(0, 0.15, n_patients)               # Noise
)

# Convert to binary (top 45% have disease)
threshold = np.percentile(risk_score, 55)
df['heart_disease'] = (risk_score > threshold).astype(int)

print("✓ Heart disease dataset created!")
print(f"\nDataset shape: {df.shape}")
print(f"Patients: {len(df)}")
print(f"Features: {df.shape[1] - 1}")
print(f"\nDisease prevalence: {df['heart_disease'].mean()*100:.1f}%")

In [ ]:
# Display sample data
print("Sample Patient Data:")
display(df.head(10))

# Statistical summary
print("\nDataset Statistics:")
display(df.describe())

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Class distribution
df['heart_disease'].value_counts().plot(
    kind='bar', 
    ax=axes[0], 
    color=['#4ECDC4', '#FF6B6B'], 
    alpha=0.8, 
    edgecolor='black'
)
axes[0].set_title('Heart Disease Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Heart Disease', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['No Disease', 'Disease'], rotation=0)
axes[0].grid(axis='y', alpha=0.3)

# Age distribution by disease
df[df['heart_disease']==0]['age'].hist(bins=20, alpha=0.6, label='No Disease', color='#4ECDC4', ax=axes[1])
df[df['heart_disease']==1]['age'].hist(bins=20, alpha=0.6, label='Disease', color='#FF6B6B', ax=axes[1])
axes[1].set_title('Age Distribution by Disease Status', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Age', fontsize=12)
axes[1].set_ylabel('Frequency', fontsize=12)
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 3: Prepare Data

In [ ]:
# Separate features and target
X = df.drop('heart_disease', axis=1)
y = df['heart_disease']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")
print(f"\nFeature columns: {list(X.columns)}")

In [ ]:
# Split into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("✓ Data split complete!\n")
print(f"Training samples: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Testing samples: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")

# Scale features (important for some algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("\n✓ Features scaled using StandardScaler")

## Step 4: Define All Classification Algorithms

We'll compare 8 different algorithms!

In [ ]:
print("\n" + "="*70)
print("DEFINING CLASSIFICATION ALGORITHMS")
print("="*70)

# Define algorithms to compare
algorithms = {
    'Logistic Regression': {
        'model': LogisticRegression(max_iter=1000, random_state=42),
        'needs_scaling': True,
        'description': 'Linear model for binary classification'
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(max_depth=10, random_state=42),
        'needs_scaling': False,
        'description': 'Tree-based rules for classification'
    },
    'Random Forest': {
        'model': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
        'needs_scaling': False,
        'description': 'Ensemble of decision trees'
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
        'needs_scaling': False,
        'description': 'Sequential tree building for errors'
    },
    'SVM': {
        'model': SVC(kernel='rbf', probability=True, random_state=42),
        'needs_scaling': True,
        'description': 'Support Vector Machine with RBF kernel'
    },
    'K-Nearest Neighbors': {
        'model': KNeighborsClassifier(n_neighbors=5),
        'needs_scaling': True,
        'description': 'Majority vote from K nearest neighbors'
    },
    'Naive Bayes': {
        'model': GaussianNB(),
        'needs_scaling': False,
        'description': 'Probabilistic classifier using Bayes theorem'
    },
    'AdaBoost': {
        'model': AdaBoostClassifier(n_estimators=100, random_state=42),
        'needs_scaling': False,
        'description': 'Adaptive boosting ensemble method'
    }
}

print(f"\n✓ {len(algorithms)} algorithms defined:")
for name, info in algorithms.items():
    scaling = "Needs scaling" if info['needs_scaling'] else "Scale-invariant"
    print(f"  • {name:25} - {scaling}")

print("\n💡 Some algorithms need scaled features, others don't!")

## Step 5: Train and Evaluate All Algorithms

Let's systematically train and evaluate each algorithm.

In [ ]:
print("\n" + "="*70)
print("TRAINING AND EVALUATING ALL ALGORITHMS")
print("="*70)

results = []

for name, info in algorithms.items():
    print(f"\n[{list(algorithms.keys()).index(name)+1}/{len(algorithms)}] Training {name}...")
    
    # Choose appropriate data (scaled or unscaled)
    if info['needs_scaling']:
        X_tr, X_te = X_train_scaled, X_test_scaled
        print(f"    Using SCALED features")
    else:
        X_tr, X_te = X_train.values, X_test.values
        print(f"    Using UNSCALED features")
    
    # Train
    start_time = time.time()
    model = info['model']
    model.fit(X_tr, y_train)
    training_time = time.time() - start_time
    
    # Predict
    start_time = time.time()
    y_pred = model.predict(X_te)
    prediction_time = time.time() - start_time
    
    # Get probabilities for ROC
    if hasattr(model, 'predict_proba'):
        y_pred_proba = model.predict_proba(X_te)[:, 1]
    else:
        y_pred_proba = model.decision_function(X_te)
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results.append({
        'Algorithm': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc,
        'Training Time (s)': training_time,
        'Prediction Time (s)': prediction_time,
        'Model': model,
        'Predictions': y_pred,
        'Probabilities': y_pred_proba
    })
    
    print(f"    ✓ Accuracy: {accuracy*100:.2f}% | ROC-AUC: {roc_auc:.3f} | Time: {training_time:.3f}s")

# Create results DataFrame
results_df = pd.DataFrame(results)

print("\n" + "="*70)
print("✓ ALL ALGORITHMS TRAINED AND EVALUATED!")
print("="*70)

## Step 6: Comparison Table

In [ ]:
# Display comparison table
print("\nCOMPREHENSIVE MODEL COMPARISON:")
print("="*90)

comparison_table = results_df[[
    'Algorithm', 'Accuracy', 'Precision', 'Recall', 
    'F1-Score', 'ROC-AUC', 'Training Time (s)'
]].copy()

# Sort by ROC-AUC (best overall metric for binary classification)
comparison_table = comparison_table.sort_values('ROC-AUC', ascending=False)

display(comparison_table)

# Find best performers
best_accuracy = comparison_table.loc[comparison_table['Accuracy'].idxmax()]
best_roc_auc = comparison_table.loc[comparison_table['ROC-AUC'].idxmax()]
best_f1 = comparison_table.loc[comparison_table['F1-Score'].idxmax()]
fastest = comparison_table.loc[comparison_table['Training Time (s)'].idxmin()]

print("\n🏆 BEST PERFORMERS:")
print(f"  • Best Accuracy:    {best_accuracy['Algorithm']} ({best_accuracy['Accuracy']*100:.2f}%)")
print(f"  • Best ROC-AUC:     {best_roc_auc['Algorithm']} ({best_roc_auc['ROC-AUC']:.3f})")
print(f"  • Best F1-Score:    {best_f1['Algorithm']} ({best_f1['F1-Score']:.3f})")
print(f"  • Fastest Training: {fastest['Algorithm']} ({fastest['Training Time (s)']:.4f}s)")

## Step 7: Visualize Performance Metrics

In [ ]:
# Create comprehensive comparison visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Accuracy comparison
sorted_by_acc = comparison_table.sort_values('Accuracy', ascending=True)
axes[0, 0].barh(sorted_by_acc['Algorithm'], sorted_by_acc['Accuracy']*100, 
                color='#FF6B6B', alpha=0.8, edgecolor='black')
axes[0, 0].set_xlabel('Accuracy (%)', fontsize=11, fontweight='bold')
axes[0, 0].set_title('Accuracy Comparison', fontsize=13, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)
axes[0, 0].set_xlim([50, 100])

# 2. ROC-AUC comparison
sorted_by_auc = comparison_table.sort_values('ROC-AUC', ascending=True)
axes[0, 1].barh(sorted_by_auc['Algorithm'], sorted_by_auc['ROC-AUC'], 
                color='#4ECDC4', alpha=0.8, edgecolor='black')
axes[0, 1].set_xlabel('ROC-AUC Score', fontsize=11, fontweight='bold')
axes[0, 1].set_title('ROC-AUC Comparison', fontsize=13, fontweight='bold')
axes[0, 1].grid(axis='x', alpha=0.3)
axes[0, 1].set_xlim([0.5, 1.0])

# 3. Precision vs Recall
axes[1, 0].scatter(comparison_table['Recall'], comparison_table['Precision'], 
                   s=200, alpha=0.7, c=range(len(comparison_table)), cmap='viridis', 
                   edgecolor='black', linewidth=2)
for idx, row in comparison_table.iterrows():
    axes[1, 0].annotate(row['Algorithm'], 
                        (row['Recall'], row['Precision']), 
                        fontsize=8, ha='right')
axes[1, 0].set_xlabel('Recall', fontsize=11, fontweight='bold')
axes[1, 0].set_ylabel('Precision', fontsize=11, fontweight='bold')
axes[1, 0].set_title('Precision vs Recall Trade-off', fontsize=13, fontweight='bold')
axes[1, 0].grid(alpha=0.3)
axes[1, 0].set_xlim([0.5, 1.0])
axes[1, 0].set_ylim([0.5, 1.0])

# 4. Training time comparison
sorted_by_time = comparison_table.sort_values('Training Time (s)', ascending=True)
axes[1, 1].barh(sorted_by_time['Algorithm'], sorted_by_time['Training Time (s)'], 
                color='#45B7D1', alpha=0.8, edgecolor='black')
axes[1, 1].set_xlabel('Training Time (seconds)', fontsize=11, fontweight='bold')
axes[1, 1].set_title('Training Speed Comparison', fontsize=13, fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.suptitle('Comprehensive Model Comparison - Heart Disease Classification', 
             fontsize=16, fontweight='bold', y=1.00)
plt.tight_layout()
plt.show()

## Step 8: ROC Curves for All Models

ROC curves show the trade-off between True Positive Rate and False Positive Rate.

In [ ]:
# Plot ROC curves for all algorithms
plt.figure(figsize=(12, 8))

colors = plt.cm.tab10(np.linspace(0, 1, len(results_df)))

for idx, row in results_df.iterrows():
    fpr, tpr, _ = roc_curve(y_test, row['Probabilities'])
    auc = row['ROC-AUC']
    
    plt.plot(fpr, tpr, 
             label=f"{row['Algorithm']} (AUC = {auc:.3f})",
             linewidth=2.5,
             color=colors[idx],
             alpha=0.8)

# Plot diagonal (random classifier)
plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier (AUC = 0.500)', alpha=0.5)

plt.xlabel('False Positive Rate', fontsize=13, fontweight='bold')
plt.ylabel('True Positive Rate', fontsize=13, fontweight='bold')
plt.title('ROC Curves - All Algorithms', fontsize=15, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("\n📊 ROC Curve Analysis:")
print("  • Curves closer to top-left are better")
print("  • AUC = Area Under Curve (0.5 = random, 1.0 = perfect)")
print("  • Higher AUC = better overall discrimination ability")

## Step 9: Confusion Matrices for Top 3 Models

In [ ]:
# Show confusion matrices for top 3 models by ROC-AUC
top_3 = results_df.nlargest(3, 'ROC-AUC')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Confusion Matrices - Top 3 Models', fontsize=16, fontweight='bold')

for idx, (_, row) in enumerate(top_3.iterrows()):
    cm = confusion_matrix(y_test, row['Predictions'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['No Disease', 'Disease']
    )
    disp.plot(cmap='Blues', ax=axes[idx], colorbar=False)
    axes[idx].set_title(
        f"{row['Algorithm']}\nAUC: {row['ROC-AUC']:.3f}",
        fontsize=12,
        fontweight='bold'
    )

plt.tight_layout()
plt.show()

## Step 10: Algorithm Selection Guide

When should you use which algorithm?

In [ ]:
print("\n" + "="*80)
print("ALGORITHM SELECTION GUIDE")
print("="*80)

guide = {
    'Logistic Regression': {
        'When to Use': [
            '• Need interpretable model (coefficients show feature importance)',
            '• Linear relationship between features and target',
            '• Fast training and prediction needed',
            '• Probability estimates are important'
        ],
        'Pros': 'Fast, interpretable, works well with linear relationships',
        'Cons': 'Poor with non-linear relationships, needs feature engineering'
    },
    'Decision Tree': {
        'When to Use': [
            '• Need highly interpretable model (can visualize tree)',
            '• Data has categorical variables',
            '• No need for feature scaling',
            '• Want to understand decision rules'
        ],
        'Pros': 'Very interpretable, handles mixed data types, no scaling needed',
        'Cons': 'Prone to overfitting, unstable (small data changes affect tree)'
    },
    'Random Forest': {
        'When to Use': [
            '• Want high accuracy without much tuning',
            '• Need feature importance ranking',
            '• Have enough data (works better with more samples)',
            '• Can afford slightly longer training time'
        ],
        'Pros': 'High accuracy, handles non-linear relationships, feature importance',
        'Cons': 'Slower than single trees, less interpretable, larger memory'
    },
    'Gradient Boosting': {
        'When to Use': [
            '• Need the best possible accuracy',
            '• Have time for hyperparameter tuning',
            '• Data is structured/tabular',
            '• Willing to trade interpretability for performance'
        ],
        'Pros': 'Often highest accuracy, handles complex patterns well',
        'Cons': 'Easy to overfit, requires careful tuning, slower training'
    },
    'SVM': {
        'When to Use': [
            '• High-dimensional data (many features)',
            '• Clear margin of separation exists',
            '• Small to medium dataset size',
            '• Need robust model (less affected by outliers with RBF kernel)'
        ],
        'Pros': 'Works well in high dimensions, effective with clear margins',
        'Cons': 'Requires feature scaling, slow on large datasets, needs tuning'
    },
    'K-Nearest Neighbors': {
        'When to Use': [
            '• Small dataset',
            '• Non-linear decision boundaries',
            '• No training phase needed (lazy learning)',
            '• Data is well-clustered'
        ],
        'Pros': 'Simple, no training time, works with any decision boundary',
        'Cons': 'Very slow prediction on large datasets, needs feature scaling'
    },
    'Naive Bayes': {
        'When to Use': [
            '• Text classification (spam detection, sentiment analysis)',
            '• Need very fast training and prediction',
            '• Features are relatively independent',
            '• Small dataset with high dimensions'
        ],
        'Pros': 'Extremely fast, works well with text, good with small data',
        'Cons': 'Assumes feature independence (rarely true), less accurate'
    },
    'AdaBoost': {
        'When to Use': [
            '• Want ensemble benefits with simpler base models',
            '• Have weak learners that need boosting',
            '• Need better performance than single decision trees',
            '• Less prone to overfitting than Gradient Boosting'
        ],
        'Pros': 'Good accuracy, less prone to overfitting than GB',
        'Cons': 'Sensitive to noisy data and outliers'
    }
}

# Print guide
for algo, info in guide.items():
    print(f"\n{algo.upper()}")
    print("-" * 80)
    print("When to Use:")
    for point in info['When to Use']:
        print(f"  {point}")
    print(f"\n  ✓ Pros: {info['Pros']}")
    print(f"  ✗ Cons: {info['Cons']}")

print("\n" + "="*80)

## Step 11: Final Recommendations

In [ ]:
print("\n" + "="*80)
print("FINAL RECOMMENDATIONS FOR THIS DATASET")
print("="*80)

# Get top 3 models
top_3_names = comparison_table.head(3)['Algorithm'].tolist()
top_3_auc = comparison_table.head(3)['ROC-AUC'].tolist()

print(f"\n🏆 TOP 3 ALGORITHMS FOR HEART DISEASE PREDICTION:\n")
for i, (name, auc) in enumerate(zip(top_3_names, top_3_auc), 1):
    print(f"{i}. {name}")
    print(f"   ROC-AUC: {auc:.3f}")
    
    # Add specific recommendation
    if 'Gradient Boosting' in name or 'Random Forest' in name:
        print(f"   💡 Best for: Maximum accuracy")
    elif 'Logistic' in name:
        print(f"   💡 Best for: Interpretability and speed")
    elif 'SVM' in name:
        print(f"   💡 Best for: Robust predictions")
    print()

print("\n💼 DEPLOYMENT CONSIDERATIONS:\n")
print("For Production (Medical Application):")
print(f"  • Primary: {top_3_names[0]} (best performance)")
print(f"  • Backup: Logistic Regression (interpretable for doctors)")
print(f"  • Why both? {top_3_names[0]} for accuracy, Logistic for explanation")

print("\n⚠️ IMPORTANT FOR MEDICAL ML:")
print("  • High RECALL is critical (don't miss disease cases!)")
print("  • Model must be interpretable for medical professionals")
print("  • Consider ensemble: use multiple models and vote")
print("  • Always have human doctor review before final decision")

print("\n" + "="*80)

## Step 12: Summary and Key Takeaways

In [ ]:
print("\n" + "="*80)
print("PROJECT SUMMARY - SYSTEMATIC MODEL COMPARISON")
print("="*80)

print("\n✅ WHAT YOU ACCOMPLISHED:")
print("  1. Created synthetic heart disease dataset")
print("  2. Prepared data with proper scaling")
print(f"  3. Trained and evaluated {len(algorithms)} different algorithms")
print("  4. Compared all metrics: accuracy, precision, recall, F1, ROC-AUC")
print("  5. Created comprehensive comparison visualizations")
print("  6. Plotted ROC curves for all models")
print("  7. Analyzed confusion matrices")
print("  8. Built algorithm selection guide")
print("  9. Made deployment recommendations")

print("\n🎯 KEY LEARNINGS:")
print("  • No single 'best' algorithm - it depends on requirements")
print("  • Always compare multiple algorithms systematically")
print("  • Consider accuracy, speed, interpretability, and robustness")
print("  • ROC-AUC is often better than accuracy for imbalanced data")
print("  • Some algorithms need scaling, others don't")
print("  • Ensemble methods (RF, GB) often win on accuracy")
print("  • Simple models (Logistic, Naive Bayes) win on speed and interpretability")

print("\n📊 FINAL RESULTS:")
print(f"  • Dataset: {len(df)} patients")
print(f"  • Features: {X.shape[1]}")
print(f"  • Algorithms compared: {len(algorithms)}")
print(f"  • Best algorithm: {top_3_names[0]}")
print(f"  • Best ROC-AUC: {top_3_auc[0]:.3f}")
print(f"  • Fastest training: {fastest['Algorithm']} ({fastest['Training Time (s)']:.4f}s)")

print("\n🎓 DECISION FRAMEWORK:")
print("  Start with these questions:")
print("  1. Do I need interpretability? → Logistic Regression or Decision Tree")
print("  2. Do I have lots of data? → Random Forest or Gradient Boosting")
print("  3. Is speed critical? → Naive Bayes or Logistic Regression")
print("  4. Do I have high-dimensional data? → SVM")
print("  5. Is it text data? → Naive Bayes")
print("  6. Want best accuracy, have time to tune? → Gradient Boosting")

print("\n💡 BEST PRACTICE:")
print("  ALWAYS try multiple algorithms!")
print("  • Start with 3-5 different types")
print("  • Compare on YOUR specific data")
print("  • Consider your constraints (time, interpretability, etc.)")
print("  • Don't assume what works on one dataset works on another")

print("\n🚀 YOU'VE COMPLETED MODULE 2!")
print("  You now know:")
print("  ✓ Binary and multi-class classification")
print("  ✓ Text classification (spam detection)")
print("  ✓ Business applications (churn prediction)")
print("  ✓ Feature engineering techniques")
print("  ✓ Systematic model comparison")
print("  ✓ When to use which algorithm")

print("\n" + "="*80)
print("🎉 CONGRATULATIONS! YOU'RE NOW A CLASSIFICATION EXPERT!")
print("="*80)

---

## AI Prompts for This Notebook

### Prompt 1: Dataset Creation
```
Create a synthetic heart disease dataset with 800 patients in Python. Include 
features: age, sex, chest pain type, resting BP, cholesterol, max heart rate, 
exercise angina, and oldpeak. Make disease probability increase with age, high 
cholesterol, and high blood pressure.
```

### Prompt 2: Algorithm Training
```
Train 8 classification algorithms on heart disease data: Logistic Regression, 
Decision Tree, Random Forest, Gradient Boosting, SVM, KNN, Naive Bayes, and 
AdaBoost. Calculate accuracy, precision, recall, F1-score, and ROC-AUC for each. 
Time the training and prediction.
```

### Prompt 3: Comparison Visualization
```
Create a 2x2 subplot comparing 8 classification algorithms. Show: (1) accuracy 
horizontal bar chart, (2) ROC-AUC horizontal bar chart, (3) precision vs recall 
scatter plot, (4) training time comparison. Make it professional and publication-ready.
```

### Prompt 4: ROC Curves
```
Plot ROC curves for 8 classification algorithms on the same graph. Each should 
have a different color and show its AUC score in the legend. Include the diagonal 
line for random classifier. Make legend readable.
```

### Prompt 5: Algorithm Selection Guide
```
Create a comprehensive guide for when to use each classification algorithm. For 
Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, SVM, KNN, 
Naive Bayes, and AdaBoost, list: when to use, pros, and cons. Format clearly.
```

### Prompt 6: Deployment Recommendations
```
Based on model comparison results for heart disease prediction, provide deployment 
recommendations. Consider accuracy, interpretability, and medical application 
requirements. Suggest primary and backup models with justification.
```

---

**End of Notebook**

**Created:** 2026-01-06  
**Course:** ML for Engineers - Module 2  
**Project:** Systematic Model Comparison  
**Algorithms Compared:** 8 ✅